# Lost Wells: Appalachian U-Net, from first principles

This notebook is deliberately staged. Do not use a later stage until the preceding check passes.

A digital image is a grid of pixels. A segmentation model assigns a probability to every pixel: here, ‘does this pixel belong to a printed well symbol?’ Connected positive pixels form a detection. Georeferencing converts its pixel location into longitude/latitude. Finally, a 100 m comparison against state well registries separates ordinary documented symbols from candidate undocumented wells.

We begin with **inference**, meaning use of LBNL’s already-trained parameters. Training changes those parameters from labeled examples. We will fine-tune only after inference is measured on manually reviewed Appalachian examples; training on an incomplete well database would incorrectly teach the network that undocumented symbols are background.

## 1. Select a GPU and install the legacy model environment
In Colab choose **Runtime → Change runtime type → T4 GPU**. Run the next cell once, then use **Runtime → Restart session**. The restart is necessary because NumPy and Keras may already be imported by Colab.

In [ ]:
import os, subprocess, sys, importlib.metadata as metadata
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['SM_FRAMEWORK'] = 'tf.keras'
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'gdal-bin', 'libgdal-dev'], check=True)
tf_version = metadata.version('tensorflow')
print('Installed TensorFlow:', tf_version)
assert tf_version.startswith('2.20.'), f'This cell is pinned for Colab TF 2.20, found {tf_version}'
def pip_install(*packages, no_deps=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade']
    if no_deps: cmd.append('--no-deps')
    cmd.extend(packages)
    print('\nRUN:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-12000:])
    if result.returncode: raise RuntimeError(f'pip failed with exit code {result.returncode}')
pip_install('numpy==1.26.4', 'opencv-python-headless==4.10.0.84')
pip_install('tf_keras==2.20.1')
pip_install('keras_applications==1.0.8', 'image-classifiers==1.0.0', 'efficientnet==1.1.1', 'segmentation-models==1.0.1', no_deps=True)
pip_install('simplekml==1.3.6', 'geopandas', 'requests')
gdal_version = subprocess.check_output(['gdal-config', '--version'], text=True).strip()
pip_install(f'GDAL=={gdal_version}')
print('Install complete. Now restart the Colab session, then continue below.')

Installed TensorFlow: 2.20.0

RUN: /usr/bin/python3 -m pip install --upgrade numpy==1.26.4 opencv-python-headless==4.10.0.84


RUN: /usr/bin/python3 -m pip install --upgrade tf_keras==2.20.1


RUN: /usr/bin/python3 -m pip install --upgrade --no-deps keras_applications==1.0.8 image-classifiers==1.0.0 efficientnet==1.1.1 segmentation-models==1.0.1


RUN: /usr/bin/python3 -m pip install --upgrade simplekml==1.3.6 geopandas requests


## 2. Mount persistent storage and obtain the code
Colab’s `/content` disk disappears when the session ends. Google Drive will hold the model, manifests, well registries, and GeoJSON checkpoints. Individual map images remain temporary.

In [ ]:

import os, subprocess, sys
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['SM_FRAMEWORK'] = 'tf.keras'
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/lostwells'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/ShrishPremkrishna/lostwells.git', REPO], check=True)
UNET = REPO + '/UNET'
WORK = '/content/drive/MyDrive/lostwells_unet'
os.makedirs(WORK, exist_ok=True)
os.chdir(UNET)
print('code:', UNET, '\npersistent work:', WORK)

In [ ]:
import tensorflow as tf, segmentation_models as sm, numpy as np
from osgeo import gdal
print('TensorFlow', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU: change the Colab runtime type.'
assert int(np.__version__.split('.')[0]) < 2
print('Environment gate: PASS')

## 3. Download the released model and prove reproduction
This is a scientific control. Before asking whether the model transfers to Appalachia, verify that our implementation reproduces released California results on the supplied map. A model that loads is not necessarily a model that is preprocessed correctly.

In [ ]:
from pathlib import Path
import subprocess, zipfile
LBNL = Path(WORK) / 'lbnl'
downloader = Path(UNET) / 'scripts' / 'get_model.py'
assert downloader.exists(), f'Missing {downloader}; the GitHub checkout does not contain the UNET scripts'
needed = ['unet_model.h5', 'CA_OilCenter', 'CalGEM_AllWells', 'found_potential_UOWs']
result = subprocess.run([sys.executable, str(downloader), '--out', str(LBNL), '--only', *needed], cwd=UNET, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'Model downloader failed with exit code {result.returncode}; read the [model] message above')
for archive in LBNL.glob('*.zip'):
    with zipfile.ZipFile(archive) as z: z.extractall(LBNL / archive.stem)
model = next(LBNL.rglob('unet_model.h5'))
sample_tif = next(LBNL.rglob('*_geo.tif'))
calgem = next(LBNL.rglob('*CalGEM*.csv'))
published = next(p for p in LBNL.rglob('*.csv') if 'Kern_CA_UOWs' in p.name)
print(model, sample_tif, calgem, published, sep='\n')

In [ ]:
subprocess.run([sys.executable, 'scripts/validate_kern.py', '--quads', str(sample_tif.parent), '--model', str(model), '--documented', str(calgem), '--published', str(published), '--batch-size', '16'], check=True)
print('Read the final validation line. Continue only if it says PASS.')

## 4. Build the core Appalachian first-pass manifest
A quadrangle can have several historical editions. The newest 1947–1992 edition is the economical first pass: approximately 2,700 maps across PA/OH/WV/KY, rather than about 8,000 editions. Older editions become a later recovery pass because an abandoned-well symbol may disappear from a newer map.

In [ ]:
MAPS = Path(WORK) / 'maps'
subprocess.run([sys.executable, 'scripts/download_maps.py', '--states', 'PA', 'OH', 'WV', 'KY', '--latest-per-quad', '--manifest-only', '--out', str(MAPS), '--cache', str(Path(WORK)/'inventory')], check=True)
print('Inspect the reported map count and estimated size. No GeoTIFFs were downloaded.')

## 5. Download the documented-well baseline
These registries are not training labels in this phase. They are a spatial exclusion set: a detected map symbol within 100 m of a registry point is not called undocumented. Registry incompleteness therefore raises false candidates, so provenance matters.

In [ ]:
WELLS = Path(WORK) / 'wells'
subprocess.run([sys.executable, 'scripts/download_wells.py', '--states', 'PA', 'OH', 'WV', 'KY', '--out', str(WELLS)], check=True)
well_files = [WELLS/f'{state}.geojson' for state in ('PA','OH','WV','KY')]
missing = [str(p) for p in well_files if not p.exists()]
assert not missing, f'Missing registry outputs: {missing}. Do not run inference with incomplete dedup data.'
print(*well_files, sep='\n')

## 6. Pilot, then resume in bounded sessions
The model is loaded once. For each manifest row the runner downloads one map, predicts small tile batches, writes a persistent per-map GeoJSON—even when it has zero detections—and deletes the image. Re-running skips completed outputs. First use `--limit 5`; inspect failures and maps visually before increasing to 50–200 per session.

In [ ]:
OUT = Path(WORK) / 'outputs' / 'per_quad'
cmd = [sys.executable, 'scripts/run_batch.py', '--manifest', str(MAPS/'manifest.csv'), '--model', str(model), '--documented', *map(str, well_files), '--out-dir', str(OUT), '--scratch', '/content/lostwells_maps', '--limit', '5', '--batch-size', '16']
subprocess.run(cmd, check=True)

In [ ]:
MERGED = Path(WORK) / 'outputs' / 'appalachia_candidates.geojson'
subprocess.run([sys.executable, 'scripts/merge_detections.py', '--inputs', str(OUT), '--out', str(MERGED), '--radius-m', '60'], check=True)
print('Standalone result:', MERGED)

## 7. Where actual training begins

Training requires pairs `(image tile, correct pixel mask)`. The loss function measures disagreement; backpropagation computes how each parameter contributed; Adam makes a small update; validation data estimates performance on examples not used for updates.

Our next learning stage is therefore **manual Appalachian validation**, not immediate fine-tuning:
1. Randomly sample detections and empty-looking areas across states, map years, and scan styles.
2. Manually mark every visible well symbol, including symbols absent from registries.
3. Split by whole quadrangle—not random tiles—to prevent nearly identical neighboring tiles leaking into training and validation.
4. Measure precision, recall, and localization error for the pretrained checkpoint.
5. Fine-tune only if that measured baseline identifies a systematic transfer problem.

Do not run `make_labels.py` as if database points were truth: undocumented wells are missing from that database and would be trained as background. The repository’s fine-tuning script remains an experimental fallback; we will build a manual-label lesson after the five-map pilot is reviewed.